In [ ]:
# @title 🚀 GIGA-ALIGNER V5: Complete Suite (MuRIL + LaBSE + Strict/Soft Rescue)
# Dependencies: pip install transformers sentence-transformers scipy scikit-learn

import gc
import torch
import numpy as np
from scipy.optimize import linear_sum_assignment
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display, HTML

# ==========================================================
# 1. CONFIGURATION & HYPERPARAMETERS
# ==========================================================
CONFIG = {
    "USE_STRICT_HUNGARIAN": True,  # Forces 1-1 mapping if sentence lengths match
    "LABSE_WEIGHT": 0.65,          # Weight for Semantic Model (LaBSE)
    "MURIL_WEIGHT": 0.35,          # Weight for Syntactic Model (MuRIL)
    "MAIN_THRESHOLD": 0.55,        # Confidence threshold for main links
    "RESCUE_ORPHANS": True,        # Enable 'Soft Rescue' for grammar words
    "RESCUE_THRESHOLD": 0.35       # Lower threshold for rescuing orphans
}

# ==========================================================
# 2. MEMORY MANAGEMENT (Prevents OOM Errors)
# ==========================================================
def clean_memory():
    """Aggressively clears GPU memory before loading models."""
    for var in ['labse_model', 'muril_model', 'muril_tokenizer', 'sim_model']:
        if var in globals(): del globals()[var]
    gc.collect()
    torch.cuda.empty_cache()
    print("🧹 Memory Cleaned.")

clean_memory()

# ==========================================================
# 3. MODEL LOADER
# ==========================================================
print("⏳ Loading Models (32-bit Precision)...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# A. Semantic Model (LaBSE)
labse_model = SentenceTransformer('sentence-transformers/LaBSE').to(device)

# B. Syntactic Model (MuRIL - Google's Indian Lang Specialist)
muril_id = "google/muril-base-cased"
muril_tokenizer = AutoTokenizer.from_pretrained(muril_id)
muril_model = AutoModel.from_pretrained(muril_id, output_hidden_states=True).to(device)
muril_model.eval()
print("✅ Models Ready.\n")

# ==========================================================
# 4. OPTIMIZED EMBEDDING ENGINE
# ==========================================================
def get_optimized_embeddings(text, tokenizer, model):
    """
    Generates rich word vectors by:
    1. Summing the last 4 hidden layers (captures both syntax & meaning).
    2. Performing Mean Pooling on subwords to reconstruct whole-word vectors.
    """
    words = text.split()
    inputs = tokenizer(words, is_split_into_words=True, return_tensors="pt", padding=True).to(device)
    
    with torch.no_grad():
        out = model(**inputs)
    
    # Layer Aggregation (Sum last 4 layers)
    hidden_states = torch.stack(out.hidden_states)
    word_embeddings = torch.sum(hidden_states[-4:], dim=0) 
    
    # Subword Pooling (Reconstruct Words)
    word_ids = inputs.word_ids()
    final_vectors = []
    current_vectors = []
    current_word_idx = None
    
    for i, word_idx in enumerate(word_ids):
        if word_idx is None: continue # Skip [CLS], [SEP]
        
        if current_word_idx is None:
            current_word_idx = word_idx
            current_vectors.append(word_embeddings[0, i])
        elif word_idx == current_word_idx:
            current_vectors.append(word_embeddings[0, i])
        else:
            # Word boundary detected: Average previous vectors
            if current_vectors:
                final_vectors.append(torch.stack(current_vectors).mean(dim=0))
            current_word_idx = word_idx
            current_vectors = [word_embeddings[0, i]]
            
    # Append last word
    if current_vectors:
        final_vectors.append(torch.stack(current_vectors).mean(dim=0))
        
    return torch.stack(final_vectors).cpu().numpy()

# ==========================================================
# 5. THE GIGA-ALIGNER PIPELINE
# ==========================================================
def run_alignment(src, tgt, gold_standard=None):
    # 1. Generate Matrices
    # LaBSE Matrix
    e_emb = labse_model.encode(src.split(), convert_to_tensor=True).cpu().numpy()
    t_emb = labse_model.encode(tgt.split(), convert_to_tensor=True).cpu().numpy()
    sim_mat = cosine_similarity(e_emb, t_emb)
    
    # MuRIL Matrix (Optimized)
    src_vecs = get_optimized_embeddings(src, muril_tokenizer, muril_model)
    tgt_vecs = get_optimized_embeddings(tgt, muril_tokenizer, muril_model)
    muril_mat = cosine_similarity(src_vecs, tgt_vecs)
    
    # Ensemble (Weighted Average)
    ensemble_mat = (sim_mat * CONFIG["LABSE_WEIGHT"]) + (muril_mat * CONFIG["MURIL_WEIGHT"])
    rows, cols = ensemble_mat.shape
    final_links = set()

    # 2. Logic Branching
    # BRANCH A: STRICT HUNGARIAN (For equal length sentences)
    if CONFIG["USE_STRICT_HUNGARIAN"] and rows == cols:
        row_ind, col_ind = linear_sum_assignment(1.0 - ensemble_mat)
        for r, c in zip(row_ind, col_ind):
            final_links.add((r, c))
            
    # BRANCH B: FLEXIBLE ITERATIVE MAX (For agglutinated/unequal sentences)
    else:
        # Forward Pass
        for i in range(rows):
            j = np.argmax(ensemble_mat[i])
            if ensemble_mat[i, j] > CONFIG["MAIN_THRESHOLD"]: final_links.add((i, j))
        # Backward Pass
        for j in range(cols):
            i = np.argmax(ensemble_mat[:, j])
            if ensemble_mat[i, j] > CONFIG["MAIN_THRESHOLD"]: final_links.add((i, j))
            
    # 3. Orphan Rescue (Soft Match)
    if CONFIG["RESCUE_ORPHANS"] and not (CONFIG["USE_STRICT_HUNGARIAN"] and rows == cols):
        aligned_src = {i for i, j in final_links}
        for i in range(rows):
            if i not in aligned_src:
                best_j = np.argmax(ensemble_mat[i])
                # Rescue if score is decent (>0.35), even if not perfect
                if ensemble_mat[i][best_j] > CONFIG["RESCUE_THRESHOLD"]:
                    final_links.add((i, best_j))

    # 4. Score Calculation (AER)
    aer_score = 0.0
    if gold_standard:
        sure_gold = set()
        for p in gold_standard.split():
            s, t = map(int, p.split('-'))
            sure_gold.add((s, t))
        
        matches = sure_gold.intersection(final_links)
        denominator = len(sure_gold) + len(final_links)
        aer_score = 1.0 - (2 * len(matches) / denominator) if denominator > 0 else 0.0

    return src.split(), tgt.split(), sorted(list(final_links)), aer_score

# ==========================================================
# 6. VISUALIZATION ENGINE
# ==========================================================
def render(src, tgt, links, title, aer=None):
    width = 850; height = 180
    src_gap = (width - 100) / max(1, len(src) - 1)
    tgt_gap = (width - 100) / max(1, len(tgt) - 1)
    score_html = f'<span style="color: #666; font-size: 12px; margin-left: 10px;">(AER: {aer:.2f})</span>' if aer is not None else ""
    
    html = f"""<div style="font-family: Arial; border: 1px solid #ddd; padding: 15px; background: #fff; margin-bottom: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
        <div style="font-weight: bold; color: #2c3e50; border-bottom: 2px solid #f0f0f0; padding-bottom: 8px; margin-bottom: 15px;">
            {title} {score_html}
        </div>
        <svg width="{width}" height="{height}">"""
        
    src_x = [50 + i * src_gap for i in range(len(src))]
    tgt_x = [50 + i * tgt_gap for i in range(len(tgt))]
    
    for i, j in links:
        html += f'<path d="M{src_x[i]},50 C{src_x[i]},120 {tgt_x[j]},110 {tgt_x[j]},140" stroke="#3498db" stroke-width="2" fill="none" opacity="0.6"/>'
        
    for i, w in enumerate(src):
        html += f'<text x="{src_x[i]}" y="40" text-anchor="middle" font-size="13" font-family="Arial" fill="#34495e">{w}</text>'
        html += f'<circle cx="{src_x[i]}" cy="50" r="3" fill="#34495e"/>'
    for i, w in enumerate(tgt):
        html += f'<text x="{tgt_x[i]}" y="160" text-anchor="middle" font-size="13" font-family="Arial" fill="#e91e63">{w}</text>'
        html += f'<circle cx="{tgt_x[i]}" cy="140" r="3" fill="#e91e63"/>'
        
    html += "</svg></div>"
    return HTML(html)

# ==========================================================
# 7. GOLD STANDARD DATASETS (30 Tamil + 30 Malayalam)
# ==========================================================
TAMIL_DATASET = [
    ("Water boils at 100 degrees Celsius", "நீர் 100 டிகிரி செல்சியஸில் கொதிக்கிறது", "0-0 1-5 2-4 3-1 4-2 5-3"),
    ("The sun rises in the east", "சூரியன் கிழக்கில் உதிக்கிறது", "0-0 1-0 2-2 3-1 4-1 5-1"),
    ("Plants need sunlight to grow", "தாவரங்கள் வளர சூரிய ஒளி தேவை", "0-0 1-4 2-2 2-3 3-1 4-1"),
    ("The earth rotates around the sun", "பூமி சூரியனைச் சுற்றி சுழல்கிறது", "0-0 1-0 2-3 3-2 4-1 5-1"),
    ("Gravity pulls objects towards the earth", "ஈர்ப்பு விசை பொருட்களை பூமிக்கு இழுக்கிறது", "0-0 0-1 1-4 2-2 3-3 4-3 5-3"),
    ("Sound travels faster in water", "ஒலி நீரில் வேகமாக பரவுகிறது", "0-0 1-2 2-2 3-1 4-1 5-2"),
    ("I want to book a ticket", "நான் ஒரு டிக்கெட் பதிவு செய்ய வேண்டும்", "0-0 1-5 2-3 2-4 3-3 4-2 5-2"),
    ("Where is the nearest hospital", "மிக அருகில் உள்ள மருத்துவமனை எங்கே", "0-4 1-2 2-2 3-0 3-1 4-3"),
    ("The train arrives at 5 PM", "ரயில் மாலை 5 மணிக்கு வரும்", "0-0 1-0 2-4 3-3 4-1 5-2"),
    ("How much does this cost", "இதற்கு எவ்வளவு செலவாகும்", "0-1 1-1 2-2 3-0 4-2"),
    ("Please show me your passport", "தயவுசெய்து உங்கள் கடவுச்சீட்டை காட்டுங்கள்", "0-0 1-3 2-3 3-1 4-2"),
    ("I lost my luggage", "என் லகேஜ் தொலைந்துவிட்டது", "0-0 1-2 2-0 3-1"),
    ("Smoking is injurious to health", "புகைபிடித்தல் ஆரோக்கியத்திற்கு கேடு", "0-0 1-2 2-2 3-2 4-1 5-1"),
    ("Drink plenty of water daily", "தினமும் நிறைய தண்ணீர் குடிக்கவும்", "0-3 1-1 2-1 3-2 4-0"),
    ("Wash your hands with soap", "உங்கள் கைகளை சோப்பு போட்டு கழுவுங்கள்", "0-3 1-0 2-1 3-2 4-2 5-2"),
    ("Exercise keeps the body healthy", "உடற்பயிற்சி உடலை ஆரோக்கியமாக வைத்திருக்கிறது", "0-0 1-3 2-1 3-1 4-2"),
    ("He has a severe headache", "அவனுக்கு கடுமையான தலைவலி உள்ளது", "0-0 1-3 2-3 3-1 4-2"),
    ("Consult a doctor immediately", "உடனடியாக மருத்துவரை அணுகவும்", "0-2 1-2 2-1 3-0"),
    ("The government announced new policies", "அரசாங்கம் புதிய கொள்கைகளை அறிவித்தது", "0-0 1-0 2-3 3-1 4-2"),
    ("Elections will be held next month", "தேர்தல் அடுத்த மாதம் நடைபெறும்", "0-0 1-3 2-3 3-3 4-1 5-2"),
    ("The prime minister visited the city", "பிரதமர் நகரத்திற்கு விஜயம் செய்தார்", "0-0 1-0 2-2 2-3 3-3 4-1 5-1"),
    ("Police arrested the thief yesterday", "போலீசார் நேற்று திருடனை கைது செய்தனர்", "0-0 1-3 1-4 2-2 3-2 4-1"),
    ("Thousands of people gathered there", "ஆயிரக்கணக்கான மக்கள் அங்கு திரண்டனர்", "0-0 1-0 2-1 3-3 4-2"),
    ("Voting is our right", "வாக்களிப்பது நமது உரிமை", "0-0 1-2 2-1 3-2"),
    ("I am going to the library", "நான் நூலகத்திற்கு செல்கிறேன்", "0-0 1-2 2-2 3-1 4-1 5-1"),
    ("She is my best friend", "அவள் என் சிறந்த தோழி", "0-0 1-3 2-1 3-2 4-3"),
    ("Open the door please", "தயவுசெய்து கதவை திறக்கவும்", "0-2 1-1 2-1 3-0"),
    ("I love eating biryani", "எனக்கு பிரியாணி சாப்பிட பிடிக்கும்", "0-0 1-3 2-2 3-1"),
    ("He works in a software company", "அவர் ஒரு மென்பொருள் நிறுவனத்தில் வேலை செய்கிறார்", "0-0 1-5 1-6 2-4 3-4 4-2 5-3"),
    ("It is extremely hot today", "இன்று மிக அதிக வெப்பமாக இருக்கிறது", "0-4 1-4 2-1 2-2 3-3 4-0")
]

MALAYALAM_DATASET = [
    ("Water boils at 100 degrees Celsius", "വെള്ളം 100 ഡിഗ്രി സെൽഷ്യസിൽ തിളയ്ക്കുന്നു", "0-0 1-5 2-4 3-1 4-2 5-3"),
    ("The sun rises in the east", "സൂര്യൻ കിഴക്കു ഉദിക്കുന്നു", "0-0 1-0 2-2 3-1 4-1 5-1"),
    ("Plants need sunlight to grow", "ചെടികൾ വളരാൻ സൂര്യപ്രകാശം ആവശ്യമാണ്", "0-0 1-3 2-2 3-2 4-1"),
    ("Fish live in water", "മത്സ്യങ്ങൾ വെള്ളത്തിൽ ജീവിക്കുന്നു", "0-0 1-2 2-1 3-1"),
    ("The moon shines at night", "രാത്രിയിൽ ചന്ദ്രൻ പ്രകാശിക്കുന്നു", "0-1 1-1 2-2 3-0 4-0 5-0"),
    ("Metals conduct heat", "ലോഹങ്ങൾ താപം കടത്തിവിടുന്നു", "0-0 1-2 2-1"),
    ("Where is the railway station", "റെയിൽവേ സ്റ്റേഷൻ എവിടെയാണ്", "0-2 1-2 2-0 3-1"),
    ("I want to go to Kerala", "എനിക്ക് കേരളത്തിലേക്ക് പോകണം", "0-0 1-2 2-2 3-2 4-2 5-1"),
    ("The bus is late today", "ഇന്ന് ബസ് വൈകി", "0-1 1-1 2-2 3-2 4-0"),
    ("How far is the airport", "വിമാനത്താവളം എത്ര അകലെയാണ്", "0-1 1-2 2-2 3-2 4-0"),
    ("Please give me a ticket", "ദയവായി എനിക്ക് ഒരു ടിക്കറ്റ് തരൂ", "0-0 1-4 2-1 3-2 4-3"),
    ("The road is closed", "റോഡ് അടച്ചിരിക്കുന്നു", "0-0 1-0 2-1 3-1"),
    ("Smoking causes cancer", "പുകവലി ക്യാൻസറിന് കാരണമാകുന്നു", "0-0 1-2 2-1"),
    ("Eat healthy food everyday", "ദിവസവും ആരോഗ്യകരമായ ഭക്ഷണം കഴിക്കുക", "0-3 1-1 2-2 3-0"),
    ("He has a fever", "അവന് പനിയാണ്", "0-0 1-1 2-1 3-1"),
    ("Yoga is good for health", "യോഗ ആരോഗ്യത്തിന് നല്ലതാണ്", "0-0 1-2 2-2 3-1 4-1"),
    ("Prevention is better than cure", "ചികിത്സയേക്കാൾ നല്ലത് പ്രതിരോധമാണ്", "0-3 1-3 2-1 3-1 4-0"),
    ("Wash hands before eating", "ഭക്ഷണത്തിന് മുമ്പ് കൈ കഴുകുക", "0-2 1-1 2-1 3-0"),
    ("The minister inaugurated the bridge", "മന്ത്രി പാലം ഉദ്ഘാടനം ചെയ്തു", "0-0 1-0 2-2 2-3 3-1 4-1"),
    ("Rain continues in the state", "സംസ്ഥാനത്ത് മഴ തുടരുന്നു", "0-1 1-1 2-2 3-0 4-0 5-0"),
    ("Schools will remain closed tomorrow", "നാളെ സ്കൂളുകൾക്ക് അവധിയായിരിക്കും", "0-1 1-2 2-2 3-2 4-2 5-0"),
    ("India won the match", "ഇന്ത്യ മത്സരത്തിൽ വിജയിച്ചു", "0-0 1-2 2-1 3-1"),
    ("Prices are increasing day by day", "വില ദിവസം തോറും കൂടുന്നു", "0-0 1-3 2-3 3-1 4-2 5-2"),
    ("Democracy is important", "ജനാധിപത്യം പ്രധാനമാണ്", "0-0 1-1 2-1"),
    ("I am reading a book", "ഞാൻ ഒരു പുസ്തകം വായിക്കുന്നു", "0-0 1-4 2-4 3-1 4-2"),
    ("She likes to sing songs", "അവൾക്ക് പാട്ടുകൾ പാടാൻ ഇഷ്ടമാണ്", "0-0 1-3 2-3 3-2 4-1"),
    ("They are playing football", "അവർ ഫുട്ബോൾ കളിക്കുന്നു", "0-0 1-2 2-2 3-1"),
    ("I saw him yesterday", "ഞാൻ ഇന്നലെ അവനെ കണ്ടു", "0-0 1-3 2-2 3-1"),
    ("The bird is on the tree", "പക്ഷി മരത്തിന് മുകളിലാണ്", "0-0 1-0 2-2 3-1 4-1 5-1"),
    ("It is raining heavily", "ശക്തമായി മഴ പെയ്യുന്നു", "0-2 1-2 2-2 3-1")
]

# ==========================================================
# 8. EXECUTION LOOP & VISUALIZATION
# ==========================================================
def run_benchmark():
    print(f"\n{'='*40}")
    print("🚀 STARTING GIGA-ALIGNER BENCHMARK")
    print(f"{'='*40}\n")
    
    # Run Tamil
    tam_scores = []
    print("--- TAMIL RESULTS ---")
    # Categories: Science(0-2), Travel(6-8), Health(12-14), Politics(18-20), Daily(24-26)
    indices_to_show = [0, 6, 12, 18, 24] 
    
    for idx, (src, tgt, gold) in enumerate(TAMIL_DATASET):
        s, t, l, aer = run_alignment(src, tgt, gold)
        tam_scores.append(aer)
        if idx in indices_to_show:
            display(render(s, t, l, f"Tamil Example #{idx+1}", aer))
            
    print(f"🔴 Average Tamil AER: {sum(tam_scores)/len(tam_scores):.4f}")

    # Run Malayalam
    mal_scores = []
    print("\n--- MALAYALAM RESULTS ---")
    for idx, (src, tgt, gold) in enumerate(MALAYALAM_DATASET):
        s, t, l, aer = run_alignment(src, tgt, gold)
        mal_scores.append(aer)
        if idx in indices_to_show:
            display(render(s, t, l, f"Malayalam Example #{idx+1}", aer))
            
    print(f"🔴 Average Malayalam AER: {sum(mal_scores)/len(mal_scores):.4f}")
    print(f"\n⭐ GRAND TOTAL SCORE: {(sum(tam_scores) + sum(mal_scores)) / 60:.4f}")

run_benchmark()